## Train + evaluate baseline MobileNetV2 (Colab)

This notebook trains and evaluates **baseline MobileNetV2** on:
- CIFAR-10
- CIFAR-100
- Tiny-ImageNet

It uses `experiments/run_train_eval.py` and writes results to `OUTPUT_ROOT/<dataset>/<model>/seed_<seed>/metrics.json`.

You need a **direct-download URL** to a Tiny-ImageNet archive (typically `tiny-imagenet-200.zip`).

In [ ]:
from pathlib import Path
import os

# If you opened this notebook from a cloned repo, set REPO_DIR accordingly.
# If you need to clone, set GIT_URL to your fork/repo and run the cell.
GIT_URL = ""  # e.g. "https://github.com/<you>/<repo>.git"
REPO_DIR = Path("/content/hybrid-mobilenetv2-dualconv-eca")

if not REPO_DIR.exists():
    if not GIT_URL:
        raise ValueError("Set GIT_URL to your repo URL, then re-run this cell.")
    os.system(f"git clone {GIT_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
print("Repo:", REPO_DIR)

# Install pinned deps
os.system("pip -q install -r requirements.txt")

In [ ]:
# Paths (feel free to change)
DATA_DIR = Path("/content/data")
OUTPUT_ROOT = Path("/content/outputs")

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Training controls
SEED = 42
MAX_EPOCHS = None  # set to an int (e.g. 1) for a quick sanity run

# Tiny-ImageNet direct download URL
TINY_IMAGENET_URL = ""  # required only for Tiny-ImageNet

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("SEED:", SEED)
print("MAX_EPOCHS:", MAX_EPOCHS)

In [ ]:
# Optional: download + extract Tiny-ImageNet into DATA_DIR
# This ensures the folder layout matches `data.preprocessing.get_tiny_imagenet_loaders`.

from data.download_tiny_imagenet import ensure_tiny_imagenet

if TINY_IMAGENET_URL:
    tiny_root = ensure_tiny_imagenet(url=TINY_IMAGENET_URL, data_dir=DATA_DIR)
    print("Tiny-ImageNet root:", tiny_root)
else:
    print("Skipping Tiny-ImageNet download (TINY_IMAGENET_URL not set).")

In [ ]:
import subprocess
import sys

def run(cfg_path: str, *, model: str = "baseline", dataset_root: Path | None = None):
    cmd = [
        sys.executable,
        "experiments/run_train_eval.py",
        "--config",
        cfg_path,
        "--output_root",
        str(OUTPUT_ROOT),
        "--model",
        model,
        "--dataset_root",
        str(dataset_root or DATA_DIR),
    ]
    if MAX_EPOCHS is not None:
        cmd += ["--max_epochs", str(int(MAX_EPOCHS))]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

# CIFAR-10
run("configs/cifar10.yaml")

In [ ]:
# CIFAR-100
run("configs/cifar100.yaml")

In [ ]:
# Tiny-ImageNet
# If you downloaded with ensure_tiny_imagenet(), it lives under DATA_DIR/tiny_imagenet/...
# You can also skip the download cell and point dataset_root directly to an extracted tiny-imagenet-200 folder.

tiny_root_guess = DATA_DIR / "tiny_imagenet"
if not TINY_IMAGENET_URL and not tiny_root_guess.exists():
    print("Set TINY_IMAGENET_URL (above) or set dataset_root to your extracted tiny-imagenet-200 folder.")
else:
    run("configs/tiny_imagenet.yaml", dataset_root=DATA_DIR)

In [ ]:
# Summarize saved metrics
import json

for metrics_path in sorted(OUTPUT_ROOT.glob("**/metrics.json")):
    data = json.loads(metrics_path.read_text())
    ds = data.get("dataset")
    model = data.get("model")
    seed = data.get("seed")
    test_top1 = data.get("test", {}).get("top1_pp")
    best_epoch = data.get("best_val", {}).get("epoch")
    print(f"{ds:12s} | {model:8s} | seed={seed} | best_epoch={best_epoch} | test_top1={test_top1:.2f}% | {metrics_path}")